# 01 — Dataset Overview

**Project:** AdIntel AI  
**Purpose:** quick portfolio-friendly overview of the synthetic advertising dataset.

This notebook checks that CSV files can be loaded, reviews entity coverage, calculates basic metric health, and highlights simple dataset sanity checks before deeper business validation.


## 1. Setup

In [ ]:

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

def find_project_root(start: Path | None = None) -> Path:
    """
    Find project root by walking upward until data/synthetic exists.
    This makes the notebook runnable from:
    - project root
    - notebooks/
    - VS Code interactive working directory
    """
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start, *start.parents]

    for candidate in candidates:
        if (candidate / "data" / "synthetic").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root. Please run this notebook from the project root "
        "or from a folder inside the project. Expected folder: data/synthetic/"
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "synthetic"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir    : {DATA_DIR}")

def read_csv_required(filename: str, parse_dates: list[str] | None = None) -> pd.DataFrame:
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

    df = pd.read_csv(path)

    for col in parse_dates or []:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    return df

def safe_divide(numerator, denominator):
    denominator = pd.Series(denominator).replace(0, np.nan)
    return numerator / denominator

def pct_change(post, base):
    if pd.isna(base) or base == 0:
        return np.nan
    return post / base - 1

def fmt_pct(x):
    return "n/a" if pd.isna(x) else f"{x:.2%}"

def fmt_num(x):
    if pd.isna(x):
        return "n/a"
    if abs(x) >= 1_000_000:
        return f"{x/1_000_000:.2f}M"
    if abs(x) >= 1_000:
        return f"{x/1_000:.2f}K"
    return f"{x:,.2f}"

def add_period_by_date(df: pd.DataFrame, date_col: str = "date") -> tuple[pd.DataFrame, pd.Timestamp]:
    """
    Split data dynamically:
    - first half of available date range = baseline
    - second half of available date range = decline
    """
    if date_col not in df.columns:
        raise KeyError(f"Column `{date_col}` not found.")

    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col], errors="coerce")

    min_date = out[date_col].min()
    max_date = out[date_col].max()

    if pd.isna(min_date) or pd.isna(max_date):
        raise ValueError(f"Column `{date_col}` does not contain valid dates.")

    midpoint = min_date + (max_date - min_date) / 2
    out["period"] = np.where(out[date_col] <= midpoint, "baseline", "decline")

    return out, midpoint

def require_columns(df: pd.DataFrame, cols: list[str], table_name: str = "dataframe"):
    missing = [col for col in cols if col not in df.columns]
    if missing:
        raise KeyError(f"{table_name} is missing required columns: {missing}")

def weighted_rate(df: pd.DataFrame, numerator_col: str, denominator_col: str) -> float:
    if numerator_col not in df.columns or denominator_col not in df.columns:
        return np.nan
    den = df[denominator_col].sum()
    if den == 0:
        return np.nan
    return df[numerator_col].sum() / den

def chart_bar(df, x_col, y_col, title, xlabel=None, ylabel=None, rotation=0):
    ax = df.plot(kind="bar", x=x_col, y=y_col, figsize=(9, 5), legend=False)
    ax.set_title(title)
    ax.set_xlabel(xlabel or x_col)
    ax.set_ylabel(ylabel or y_col)
    plt.xticks(rotation=rotation)
    plt.tight_layout()
    plt.show()

def chart_line(df, x_col, y_col, title, xlabel=None, ylabel=None):
    ax = df.plot(kind="line", x=x_col, y=y_col, figsize=(11, 5), legend=False)
    ax.set_title(title)
    ax.set_xlabel(xlabel or x_col)
    ax.set_ylabel(ylabel or y_col)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 2. Load CSV Files

In [ ]:

files = {
    "advertisers": "advertisers.csv",
    "campaigns": "campaigns.csv",
    "ad_groups": "ad_groups.csv",
    "creatives": "creatives.csv",
    "placements": "placements.csv",
    "inventory": "inventory.csv",
    "markets": "markets.csv",
    "devices": "devices.csv",
    "daily_ad_performance": "daily_ad_performance.csv",
    "video_performance": "video_performance.csv",
    "conversion_events": "conversion_events.csv",
    "billing_revenue": "billing_revenue.csv",
    "data_quality_logs": "data_quality_logs.csv",
}

date_tables = {
    "daily_ad_performance",
    "video_performance",
    "conversion_events",
    "billing_revenue",
    "inventory",
    "data_quality_logs",
}

data = {
    table: read_csv_required(filename, parse_dates=["date"] if table in date_tables else None)
    for table, filename in files.items()
}

advertisers = data["advertisers"]
campaigns = data["campaigns"]
ad_groups = data["ad_groups"]
creatives = data["creatives"]
placements = data["placements"]
inventory = data["inventory"]
markets = data["markets"]
devices = data["devices"]
perf = data["daily_ad_performance"]
video = data["video_performance"]
conversions = data["conversion_events"]
billing = data["billing_revenue"]
dq_logs = data["data_quality_logs"]

print("Loaded tables:", len(data))


## 3. Row Count and Column Overview

In [ ]:

row_overview = (
    pd.DataFrame([
        {
            "table": table,
            "rows": len(df),
            "columns": df.shape[1],
            "column_list": ", ".join(df.columns.astype(str)),
        }
        for table, df in data.items()
    ])
    .sort_values("table")
    .reset_index(drop=True)
)

row_overview


## 4. Date Range Overview

In [ ]:

date_overview = []

for table, df in data.items():
    if "date" in df.columns:
        date_overview.append({
            "table": table,
            "min_date": df["date"].min(),
            "max_date": df["date"].max(),
            "unique_dates": df["date"].nunique(),
        })

date_overview = pd.DataFrame(date_overview).sort_values("table").reset_index(drop=True)
date_overview


## 5. Entity Counts

In [ ]:

entity_keys = {
    "advertisers": "advertiser_id",
    "campaigns": "campaign_id",
    "ad_groups": "ad_group_id",
    "creatives": "creative_id",
    "placements": "placement_id",
    "markets": "market_id",
    "devices": "device_id",
}

entity_counts = pd.DataFrame([
    {
        "entity": table,
        "id_column": id_col,
        "rows": len(data[table]),
        "unique_ids": data[table][id_col].nunique() if id_col in data[table].columns else np.nan,
        "duplicate_ids": data[table][id_col].duplicated().sum() if id_col in data[table].columns else np.nan,
    }
    for table, id_col in entity_keys.items()
])

entity_counts


## 6. Daily Ads Metric Overview

In [ ]:

metric_cols = [
    "impressions",
    "clicks",
    "spend_usd",
    "publisher_revenue_usd",
    "measurable_impressions",
    "viewable_impressions",
]

require_columns(perf, ["date", *metric_cols], "daily_ad_performance")

metric_overview = perf[metric_cols].agg(["count", "sum", "mean", "min", "max"]).T.reset_index()
metric_overview = metric_overview.rename(columns={"index": "metric"})

overall_viewability = weighted_rate(perf, "viewable_impressions", "measurable_impressions")
overall_ctr = weighted_rate(perf, "clicks", "impressions")

summary_metrics = pd.DataFrame([
    {"metric": "overall_ctr", "value": overall_ctr, "formatted": fmt_pct(overall_ctr)},
    {"metric": "overall_viewability_rate", "value": overall_viewability, "formatted": fmt_pct(overall_viewability)},
    {"metric": "total_impressions", "value": perf["impressions"].sum(), "formatted": fmt_num(perf["impressions"].sum())},
    {"metric": "total_publisher_revenue_usd", "value": perf["publisher_revenue_usd"].sum(), "formatted": fmt_num(perf["publisher_revenue_usd"].sum())},
])

display(metric_overview)
summary_metrics


## 7. Basic Sanity Checks

In [ ]:

checks = []

non_negative_cols = [
    col for col in metric_cols
    if col in perf.columns
]

for col in non_negative_cols:
    checks.append({
        "check": f"{col} has no negative values",
        "failed_rows": int((perf[col] < 0).sum()),
    })

checks.extend([
    {
        "check": "clicks <= impressions",
        "failed_rows": int((perf["clicks"] > perf["impressions"]).sum()),
    },
    {
        "check": "viewable_impressions <= measurable_impressions",
        "failed_rows": int((perf["viewable_impressions"] > perf["measurable_impressions"]).sum()),
    },
    {
        "check": "impressions > 0",
        "failed_rows": int((perf["impressions"] <= 0).sum()),
    },
])

if "ctr" in perf.columns:
    checks.append({
        "check": "ctr between 0 and 1",
        "failed_rows": int(((perf["ctr"] < 0) | (perf["ctr"] > 1)).sum()),
    })

if "viewability_rate" in perf.columns:
    checks.append({
        "check": "viewability_rate between 0 and 1",
        "failed_rows": int(((perf["viewability_rate"] < 0) | (perf["viewability_rate"] > 1)).sum()),
    })

sanity_checks = pd.DataFrame(checks)
sanity_checks["status"] = np.where(sanity_checks["failed_rows"] == 0, "PASS", "CHECK")
sanity_checks


## 8. Simple Dataset Visualizations

In [ ]:

daily = (
    perf.groupby("date", as_index=False)
    .agg(
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum"),
        publisher_revenue_usd=("publisher_revenue_usd", "sum"),
        measurable_impressions=("measurable_impressions", "sum"),
        viewable_impressions=("viewable_impressions", "sum"),
    )
)

daily["viewability_rate"] = daily["viewable_impressions"] / daily["measurable_impressions"].replace(0, np.nan)

chart_line(daily, "date", "impressions", "Daily Impressions", ylabel="Impressions")
chart_line(daily, "date", "publisher_revenue_usd", "Daily Publisher Revenue", ylabel="Publisher Revenue USD")
chart_line(daily, "date", "viewability_rate", "Daily Viewability Rate", ylabel="Viewability Rate")


## 9. Notebook Summary

In [ ]:

summary = pd.DataFrame([
    {"item": "CSV tables loaded", "value": len(data)},
    {"item": "Daily performance rows", "value": len(perf)},
    {"item": "Date range", "value": f"{perf['date'].min().date()} to {perf['date'].max().date()}"},
    {"item": "Overall viewability", "value": fmt_pct(overall_viewability)},
    {"item": "Sanity check status", "value": "PASS" if (sanity_checks["status"] == "PASS").all() else "CHECK"},
])

summary
